# Longitudinal face-aging DataLoaders

Este notebook solo prepara y verifica los datos. Para usar el dataset completo en el servidor, cambia `DATASET_ROOT`; no hay que modificar el paquete ni reconstruir índices dentro de `__getitem__`.

In [1]:
from pathlib import Path
import sys

# Permite ejecutar el notebook desde notebooks/ o desde la raíz del repositorio.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data' / '__init__.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data import (
    build_face_aging_dataloaders,
    inspect_batch,
    plot_pair_grid,
    run_data_pipeline_validation,
)

In [2]:
def load_face_aging_data(dataset_root, *, image_size=256, batch_size=4, num_workers=0, seed=42, cache_dir=None, include_zero_delta_pairs=True, zero_delta_pair_prob=0.20, include_bidirectional_pairs=True, reverse_pair_prob=0.20):
    """Construye loaders portables a partir de la carpeta que contiene id_XXXX/."""
    dataset_root = Path(dataset_root).expanduser().resolve()
    cache_dir = Path(cache_dir).expanduser().resolve() if cache_dir else None

    loaders, metadata = build_face_aging_dataloaders(
        root_dir=dataset_root,
        image_size=image_size,
        batch_size=batch_size,
        num_workers=num_workers,          # 0 es ideal para depurar en Jupyter
        split_ratios=(0.80, 0.10, 0.10),
        seed=seed,
        train_pair_strategy='random_target',
        eval_pair_strategy='all',
        min_age_gap=1,
        max_age_gap=None,
        prompt_style='selfage',
        dynamic_person_word=False,
        horizontal_flip_prob=0.0,
        include_zero_delta_pairs=include_zero_delta_pairs, # train only; val/test remain longitudinal
        zero_delta_pair_prob=zero_delta_pair_prob,
        include_bidirectional_pairs=include_bidirectional_pairs, # train only; canonical forward index is retained
        reverse_pair_prob=reverse_pair_prob, # fraction reversed among non-self observations
        train_drop_last=False,            # útil con el sample pequeño; usar True al entrenar
        pin_memory=False,                 # cambiar a True si se entrena con CUDA
        manifest_path=(cache_dir / 'manifest.csv') if cache_dir else None,
        split_path=(cache_dir / 'splits.csv') if cache_dir else None,
    )
    return loaders, metadata

In [3]:
# Local: sample incluido. En el servidor, reemplazar por la ruta del dataset completo.
DATASET_ROOT = PROJECT_ROOT / 'data' / 'sample'
# DATASET_ROOT = Path('/ruta/en/el/servidor/dataset_unificado')

loaders, metadata = load_face_aging_data(
    DATASET_ROOT,
    image_size=256,
    batch_size=4,
    num_workers=0,
)
train_loader = loaders['train']
val_loader = loaders['val']
test_loader = loaders['test']

print('Manifest:', metadata['manifest_stats'])
print('Splits:', metadata['split_stats'])
print('Asignación por identidad:', metadata['split_assignments'])

Manifest: {'identities': 3, 'files_seen': 18, 'valid_images': 18, 'skipped_unparseable': 0, 'normalized_ages': 0, 'age': {'count': 18, 'min': 25.0, 'q25': 32.0, 'median': 38.0, 'mean': 39.0, 'q75': 46.0, 'q95': 53.15, 'q99': 53.83, 'max': 54.0}, 'images_per_identity': {'count': 3, 'min': 3.0, 'q25': 4.0, 'median': 5.0, 'mean': 6.0, 'q75': 7.5, 'q95': 9.5, 'q99': 9.9, 'max': 10.0}}
Splits: {'train': {'identities': 1, 'images': 5, 'dataset_observations': 4, 'valid_forward_pairs': 10}, 'val': {'identities': 1, 'images': 3, 'dataset_observations': 3, 'valid_forward_pairs': 3}, 'test': {'identities': 1, 'images': 10, 'dataset_observations': 44, 'valid_forward_pairs': 44}}
Asignación por identidad: {'id_1145': 'train', 'id_1144': 'val', 'id_1146': 'test'}


In [4]:
batch = next(iter(train_loader))
inspect_batch(batch)
# plot_pair_grid(batch, max_pairs=4);

source_image: (4, 3, 256, 256) torch.float32
target_image: (4, 3, 256, 256) torch.float32
source_age: [25, 29, 26, 31]
target_age: [32, 31, 31, 32]
delta_age: [7, 2, 5, 1]
source_prompt: ['photo of a person as 25-year-old', 'photo of a person as 29-year-old', 'photo of a person as 26-year-old', 'photo of a person as 31-year-old']
target_prompt: ['photo of a person as 32-year-old', 'photo of a person as 31-year-old', 'photo of a person as 31-year-old', 'photo of a person as 32-year-old']
person_id: ['id_1145', 'id_1145', 'id_1145', 'id_1145']
